# Lab 04 — CNN Architectures: ResNet → SE → ConvNeXt

**Deep Learning for Computer Vision**

Lab 03 gave you a CNN. This lab is about the ideas that turned CNNs from a few layers
deep into a few hundred, and where convolutional design landed after Transformers
arrived.

| Year | Model | The one idea |
|---|---|---|
| 2012 | **AlexNet** | ReLU + dropout + GPUs. Proved deep CNNs work |
| 2014 | **VGG** | Stack 3x3 convs. Depth beats filter size |
| 2014 | **GoogLeNet** | Multi-scale Inception blocks, 1x1 bottlenecks |
| 2015 | **ResNet** | *Skip connections.* Made 100+ layers trainable |
| 2017 | **SENet** | *Channel attention.* Recalibrate features for ~1% extra params |
| 2022 | **ConvNeXt** | A pure ConvNet, modernised with Transformer-era recipes |

We implement **ResNet** and the **SE block** from scratch, then load **ConvNeXt** to see
where the design space went. AlexNet, VGG and GoogLeNet are covered as history — you
should be able to read their diagrams, but nobody implements them by hand in 2026.

**Runs on:** Colab free T4, ~20 min total.

**Reading:** [ResNet](https://arxiv.org/abs/1512.03385) ·
[SENet](https://arxiv.org/abs/1709.01507) ·
[ConvNeXt](https://arxiv.org/abs/2201.03545) ·
[VGG](https://arxiv.org/abs/1409.1556) ·
[GoogLeNet](https://arxiv.org/abs/1409.4842)

## 0. Setup

In [ ]:
!pip install -q timm torchinfo

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt, time, numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

# T4 is Turing (sm_75): fp16 autocast works, bf16 does NOT. Use fp16 + GradScaler.
AMP_DTYPE = torch.float16
torch.manual_seed(0)

In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

train_set = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,
                                           num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False,
                                           num_workers=2, pin_memory=True)

CLASSES = train_set.classes
print(len(train_set), "train /", len(test_set), "test")
print(CLASSES)

A single training loop we reuse for every architecture in this lab. Two things make it
Colab-friendly: **mixed precision** (roughly 2x faster on a T4) and a **one-cycle
schedule**, which is what modern recipes actually use.

In [ ]:
def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total


def train(model, epochs=5, lr=1e-3, label="model", loader=None):
    loader = loader or train_loader
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(loader), pct_start=0.25)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {"loss": [], "acc": []}
    t0 = time.time()

    for ep in range(epochs):
        model.train(); running = 0.0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * y.size(0)
        acc = evaluate(model, test_loader)
        hist["loss"].append(running / len(loader.dataset))
        hist["acc"].append(acc)
        print("  [%s] epoch %d/%d  loss %.3f  test acc %.4f"
              % (label, ep + 1, epochs, hist["loss"][-1], acc))

    print("  [%s] done in %.0fs | best acc %.4f" % (label, time.time() - t0, max(hist["acc"])))
    return hist


def n_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

---
# 1. The problem ResNet solved

![degradation](img/degradation.png)

Intuition says a 56-layer network can do at least as well as a 20-layer one — the extra
layers could just learn the identity. In practice, before 2015, they did **not**. Deeper
plain networks had *higher training error*. This is the **degradation problem**, and it
is not overfitting: the model fails to fit even the data it was given.

The cause is optimisation. Learning an identity mapping through a stack of conv+ReLU
layers is hard. ResNet's fix is almost embarrassingly simple: change what the layer has
to learn.

![residual](img/residual.png)

Instead of learning H(x) directly, learn the **residual** F(x) = H(x) − x and add the
input back:

**H(x) = F(x) + x**

Now "do nothing" means driving F to zero — which weight decay does for free. The skip
connection also gives gradients an unobstructed path back to early layers, because the
derivative of x with respect to x is 1.

Below is a *plain* block — a residual block with the skip deliberately removed — so you
can measure the degradation problem yourself in Exercise 1.

In [ ]:
class PlainBlock(nn.Module):
    # A residual block with the skip connection REMOVED.
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(x)))


def make_plain_net(depth_per_stage, widths=(64, 128, 256), num_classes=10):
    layers = [nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU()]
    cin = 64
    for i, w in enumerate(widths):
        for j in range(depth_per_stage):
            layers.append(PlainBlock(cin, w, stride=2 if (j == 0 and i > 0) else 1))
            cin = w
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(cin, num_classes)]
    return nn.Sequential(*layers)


print("plain net (depth 2) params (M):", round(n_params(make_plain_net(2)), 2))

---
# 2. ResNet from scratch

![residual blocks](img/residualblocks.png)

Two block designs. **BasicBlock** (two 3x3 convs) is used in ResNet-18/34.
**Bottleneck** (1x1 → 3x3 → 1x1) is used in ResNet-50/101/152: the 1x1 convs squeeze the
channel count down before the expensive 3x3 and expand it back after, so you get more
depth for the same FLOPs.

One detail that trips everyone up: when a block changes the spatial size or the channel
count, x and F(x) no longer have the same shape, so the skip path needs a **downsample**
— a 1x1 conv with matching stride.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, cin, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return F.relu(out + identity)          # <-- the whole point


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, cin, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, planes, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return F.relu(out + identity)

In [ ]:
class ResNet(nn.Module):
    # CIFAR variant: 3x3 stem, no max-pool. A 7x7 stride-2 stem destroys 32x32 inputs.
    def __init__(self, block, layers, num_classes=10, cifar_stem=True, block_fn=None):
        super().__init__()
        self.cin = 64
        self.block_fn = block_fn or (lambda b, *a, **k: b(*a, **k))

        if cifar_stem:
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, 3, 1, 1, bias=False),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        else:
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, 7, 2, 3, bias=False),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))

        self.layer1 = self._make_layer(block, 64,  layers[0], 1)
        self.layer2 = self._make_layer(block, 128, layers[1], 2)
        self.layer3 = self._make_layer(block, 256, layers[2], 2)
        self.layer4 = self._make_layer(block, 512, layers[3], 2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        # Zero-init the last BN of each block so every block starts as an exact identity.
        for m in self.modules():
            if isinstance(m, Bottleneck):
                nn.init.zeros_(m.bn3.weight)
            elif isinstance(m, BasicBlock):
                nn.init.zeros_(m.bn2.weight)

    def _make_layer(self, block, planes, blocks, stride):
        downsample = None
        if stride != 1 or self.cin != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.cin, planes * block.expansion, 1, stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion))
        layers = [self.block_fn(block, self.cin, planes, stride, downsample)]
        self.cin = planes * block.expansion
        layers += [self.block_fn(block, self.cin, planes) for _ in range(1, blocks)]
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.pool(x).flatten(1))


def resnet18(**kw): return ResNet(BasicBlock, [2, 2, 2, 2], **kw)
def resnet34(**kw): return ResNet(BasicBlock, [3, 4, 6, 3], **kw)
def resnet50(**kw): return ResNet(Bottleneck, [3, 4, 6, 3], **kw)

print("resnet18 params (M):", round(n_params(resnet18()), 2))
print("resnet50 params (M):", round(n_params(resnet50()), 2))

In [ ]:
from torchinfo import summary
summary(resnet18(), input_size=(1, 3, 32, 32), depth=2,
        col_names=("output_size", "num_params"))

In [ ]:
hist_resnet = train(resnet18(), epochs=5, label="resnet18")

---
# 3. Squeeze-and-Excitation: attention over channels

![senet](img/senet.png)

A convolution mixes information **spatially**, but applies the same fixed filter to every
image. It has no way to say *"for this particular image the edge-detector channels matter
and the colour channels don't."*

SE adds exactly that, in three steps:

1. **Squeeze** — global-average-pool each channel to a single number, giving one
   C-dimensional vector that summarises the whole image.
2. **Excite** — a tiny bottleneck MLP (C → C/r → C) with a sigmoid, producing one gate
   per channel in [0, 1].
3. **Scale** — multiply each channel by its gate.

![se blocks](img/senetblocks.png)

That is the entire mechanism, and it costs roughly **1% extra parameters**. It is also
the cleanest example of *attention* inside a purely convolutional network — worth holding
in your head when you meet self-attention in Lab 06.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        hidden = max(channels // r, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.shape
        s = x.mean(dim=(2, 3))             # squeeze : (B,C,H,W) -> (B,C)
        s = self.fc(s).view(b, c, 1, 1)    # excite  : (B,C)     -> (B,C,1,1)
        return x * s                       # scale   : broadcast over H,W

Now wrap a residual block so SE is applied to F(x) **before** the skip is added. This
ordering matters: you want to recalibrate the residual, not the identity path.

In [ ]:
class SEWrapper(nn.Module):
    def __init__(self, block, r=16):
        super().__init__()
        self.block = block
        out_ch = (block.bn3.num_features if isinstance(block, Bottleneck)
                  else block.bn2.num_features)
        self.se = SEBlock(out_ch, r)

    def forward(self, x):
        b = self.block
        identity = x if b.downsample is None else b.downsample(x)
        out = F.relu(b.bn1(b.conv1(x)))
        if isinstance(b, Bottleneck):
            out = F.relu(b.bn2(b.conv2(out)))
            out = b.bn3(b.conv3(out))
        else:
            out = b.bn2(b.conv2(out))
        out = self.se(out)                 # <-- recalibrate the residual
        return F.relu(out + identity)


def se_block_fn(block_cls, *args, **kwargs):
    return SEWrapper(block_cls(*args, **kwargs))

def se_resnet18(**kw):
    return ResNet(BasicBlock, [2, 2, 2, 2], block_fn=se_block_fn, **kw)

base, se = n_params(resnet18()), n_params(se_resnet18())
print("resnet18    : %.2f M" % base)
print("se_resnet18 : %.2f M  (+%.1f%%)" % (se, 100 * (se / base - 1)))

In [ ]:
hist_se = train(se_resnet18(), epochs=5, label="se_resnet18")

---
# 4. ConvNeXt: what a ConvNet looks like after Transformers

When ViT and Swin started beating ResNets, the obvious conclusion was "attention beats
convolution". [ConvNeXt](https://arxiv.org/abs/2201.03545) tested that directly: start
from ResNet-50, apply the *training recipe and design choices* that arrived with
Transformers one at a time, and measure how much of the gap closes.

Almost all of it did. The changes, roughly in order of impact:

| Change | From | To |
|---|---|---|
| Training recipe | 90 epochs, SGD | 300 epochs, AdamW, heavy aug, mixup |
| Stage compute ratio | (3,4,6,3) | (3,3,9,3), like Swin |
| Stem | 7x7 s2 + maxpool | 4x4 s4 "patchify" |
| Convolution | standard 3x3 | **depthwise 7x7** (large kernel) |
| Block shape | bottleneck (narrow middle) | **inverted** bottleneck (wide middle) |
| Normalisation | BatchNorm, one per conv | **LayerNorm**, fewer |
| Activation | ReLU, one per conv | **GELU**, fewer |

The lesson is not "convolutions win" or "attention wins". It is that **a large fraction
of reported architectural gains are actually training-recipe gains**. Whenever a paper
claims a new block beats a baseline, check whether both were trained the same way.
Lab 05 makes this concrete.

Let's build one ConvNeXt block to feel the shape difference, then load the real thing.

In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, expansion=4, layer_scale=1e-6):
        super().__init__()
        # 1) depthwise 7x7 -> large receptive field, cheap because groups=dim
        self.dw = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        # 2) inverted bottleneck: expand 4x then project back, exactly like a Transformer MLP
        self.pw1 = nn.Linear(dim, expansion * dim)
        self.act = nn.GELU()
        self.pw2 = nn.Linear(expansion * dim, dim)
        # 3) LayerScale: start each block near zero. Same spirit as ResNet's zero-init BN.
        self.gamma = nn.Parameter(layer_scale * torch.ones(dim))

    def forward(self, x):
        identity = x
        x = self.dw(x)
        x = x.permute(0, 2, 3, 1)          # NCHW -> NHWC so LayerNorm/Linear act on channels
        x = self.pw2(self.act(self.pw1(self.norm(x))))
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)          # back to NCHW
        return identity + x                # still just F(x) + x


b = ConvNeXtBlock(96)
print("output:", tuple(b(torch.randn(2, 96, 16, 16)).shape))
print("block params:", sum(p.numel() for p in b.parameters()))

In [ ]:
import timm

# Pretrained ConvNeXt-Tiny, re-headed for CIFAR-10's 10 classes.
convnext = timm.create_model("convnext_tiny", pretrained=True, num_classes=10)
print("convnext_tiny params (M):", round(n_params(convnext), 2))


class Upscale(nn.Module):
    # CIFAR is 32x32; ConvNeXt's stride-4 patchify stem needs bigger inputs.
    def __init__(self, model, size=128):
        super().__init__()
        self.model, self.size = model, size

    def forward(self, x):
        return self.model(F.interpolate(x, size=self.size, mode="bilinear", align_corners=False))


hist_cnx = train(Upscale(convnext), epochs=3, lr=3e-4, label="convnext_tiny (pretrained)")

ConvNeXt reaches much higher accuracy in fewer epochs — but that comparison is **not
fair**, and it matters that you see why. It starts from ImageNet-pretrained weights; our
ResNet started from random noise. Pretraining is the single biggest lever in applied
deep learning, and it is the entire subject of Lab 05.

In [ ]:
plt.figure(figsize=(7, 4.5))
for h, name in [(hist_resnet, "resnet18 (scratch)"),
                (hist_se, "se_resnet18 (scratch)"),
                (hist_cnx, "convnext_tiny (pretrained)")]:
    plt.plot(range(1, len(h["acc"]) + 1), h["acc"], marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.grid(alpha=.3)
plt.legend(); plt.title("CIFAR-10"); plt.tight_layout(); plt.show()

print("%-32s%12s%10s" % ("model", "params (M)", "best acc"))
for m, h, name in [(resnet18(), hist_resnet, "resnet18"),
                   (se_resnet18(), hist_se, "se_resnet18"),
                   (convnext, hist_cnx, "convnext_tiny*")]:
    print("%-32s%12.2f%10.4f" % (name, n_params(m), max(h["acc"])))
print("* pretrained on ImageNet - not a like-for-like comparison")

---
# History: three you should recognise but never implement

![alexnet](img/alexnet.png)

**AlexNet (2012)** — 8 layers, 60M params. Won ImageNet by a 10-point margin and started
the field. Its contributions (ReLU over tanh, dropout, GPU training, data augmentation)
are now so standard they have become invisible.

![vgg16](img/vgg16.png)

**VGG (2014)** — the "just stack 3x3 convs" network. Two stacked 3x3 convs have the same
receptive field as one 5x5, with fewer parameters and an extra non-linearity. VGG is
absurdly heavy (138M params, most of them in the first fully-connected layer) and today
survives mainly as a *perceptual loss* backbone in generative models — you will meet it
again in Lab 10.

![inception](img/inception.png)

**GoogLeNet / Inception (2014)** — rather than choosing a filter size, run 1x1, 3x3, 5x5
and pooling in parallel and concatenate the results. The 1x1 convs are
dimensionality-reduction bottlenecks that keep it affordable. The idea that survived is
the **1x1 conv as a cheap channel mixer**, which you can see inside `Bottleneck` above.

---
# Exercises

**1. Prove the degradation problem.** Using `make_plain_net`, train plain networks at
`depth_per_stage` = 1, 3 and 6, and ResNets of matching depth. Plot final **training**
loss against depth for both families. Show the plain network gets worse with depth while
the ResNet does not. Why is it essential to plot *training* loss, not test accuracy, for
this argument to hold?

**2. Where does the skip actually help?** Take a trained `resnet18` and a plain network
of the same depth. For each, record the gradient norm at the first conv layer after one
backward pass (`p.grad.norm()` after `loss.backward()`). Compare and explain.

**3. SE ablation.** Sweep the reduction ratio `r` over 4, 8, 16, 32. Plot accuracy and
parameter count against `r`. Is 16 (the paper's choice) actually best here? Then move the
SE call to *after* the residual addition and explain what changes.

**4. Recipe vs. architecture.** Retrain `resnet18` with ConvNeXt's recipe: ReLU→GELU,
BatchNorm→GroupNorm, add mixup, train 3x longer. How much of the ConvNeXt gap can you
close *without changing the architecture*? This is the ConvNeXt paper's central
experiment, reproduced at small scale.

**5. Your own dataset.** Build a `Dataset` for a 2–5 class problem of your own (leaf
species, local food, machine parts — 100+ images per class). Fine-tune `resnet18` on it
and report a confusion matrix. **Keep this dataset — Labs 05, 07 and 08 reuse it.**